In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_col

# Не показывать Warning
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Настройки оформления графиков
plt.rcParams['figure.figsize'] = (8.5, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

CONST_NAMES = ('const', 'Intercept')


def fmt(x, decimals=3):
    """Число с заданным числом знаков; оценка, которая округляется до нуля, — с тремя значащими цифрами."""
    if pd.isna(x):
        return ''
    text = f'{x:.{decimals}f}'
    if float(text) == 0:
        return f'{x:.3g}' if x != 0 else f'{0:.{decimals}f}'
    return text


def fmt_p(p, decimals=3):
    """P-значение: меньше 0.001 печатается как «<0.001»."""
    return '<0.001' if p < 0.001 else f'{p:.{decimals}f}'


def errors_text(res, case='nom'):
    """Название типа стандартных ошибок подгонки: 'nom' — «неробастные», 'dat' — «по неробастным …»."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    if cov_type == 'nonrobust':
        return {'nom': 'неробастные стандартные ошибки',
                'dat': 'по неробастным стандартным ошибкам'}[case]
    return {'nom': f'робастные стандартные ошибки, {cov_type}',
            'dat': f'по робастным стандартным ошибкам {cov_type}'}[case]


def fit_ols(formula, data, robust=False, **kwargs):
    """МНК-подгонка; robust=True — робастные ошибки HC3. Всегда t- и F-распределения (use_t=True)."""
    cov_type = 'HC3' if robust else 'nonrobust'
    return ols(formula, data=data).fit(cov_type=cov_type, use_t=True, **kwargs)


def show_table(df, index=True, align=None):
    """Таблица pandas в Markdown; числа должны быть заранее превращены в строки через fmt."""
    display(Markdown('\n\n' + df.to_markdown(index=index, colalign=align, disable_numparse=True) + '\n\n'))


def nobs_text(res):
    """Строка «Модель была подогнана по N наблюдениям»."""
    n = int(res.nobs)
    word = 'наблюдению' if n % 10 == 1 and n % 100 != 11 else 'наблюдениям'
    return f'Модель была подогнана по {n} {word}.'


def show_coefs(res, decimals=3, show_nobs=True):
    """Оценки коэффициентов: вертикальная таблица «Переменная | Оценка» и число наблюдений."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    column = 'Оценка' if cov_type == 'nonrobust' else f'Оценка ({cov_type})'
    table = pd.DataFrame({'Переменная': res.params.index,
                          column: [fmt(b, decimals) for b in res.params]})
    text = table.to_markdown(index=False, colalign=('left', 'right'), disable_numparse=True)
    if show_nobs:
        text += '\n\n' + nobs_text(res)
    display(Markdown('\n\n' + text + '\n\n'))


def display_answer(text, use_callout=True, theme='note', title='Критическое значение'):
    """Ответ в рамке (callout): theme='note' — число, 'tip' — вывод, 'important' — условие."""
    if use_callout:
        head = f'title="{title}" ' if title else ''
        output_md = f'\n\n::: {{.callout-{theme} {head}icon="true"}}\n{text}\n:::\n\n'
    elif title:
        output_md = f'\n\n**{title}:** {text}\n\n'
    else:
        output_md = f'\n\n{text}\n\n'
    display(Markdown(output_md))


def show_conditions(level, errors=None, hypothesis=None, extra=()):
    """Блок «Условия теста»: уровень значимости (доля или список долей), тип ошибок, гипотеза формулой."""
    levels = level if isinstance(level, (list, tuple)) else [level]
    items = ['уровень значимости: **' + ' и '.join(f'{a * 100:g}%' for a in levels) + '**']
    if errors:
        items.append(f'стандартные ошибки: **{errors}**')
    hypotheses = [hypothesis] if isinstance(hypothesis, str) else list(hypothesis or [])
    if len(hypotheses) == 1:
        items.append(f'гипотеза: {hypotheses[0]}')
    else:
        items += [f'гипотеза {i}: {h}' for i, h in enumerate(hypotheses, start=1)]
    items += list(extra)
    display_answer('\n'.join(f'- {item}' for item in items), theme='important', title='Условия теста')


def signif_stars(p):
    """Коды значимости: *** p<0.01, ** p<0.05, * p<0.1."""
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''


SIGNIF_LEGEND = '*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$'


def print_regression_summary(res, decimals=3, show_signif=True):
    """Таблица t-теста: Coef., Std.Err., t-value, p-value, Signif. Заголовок называет тип ошибок."""
    title = f'**Результаты $t$-теста для коэффициентов ({errors_text(res)}):**'
    table = pd.DataFrame({
        'Coef.': [fmt(b, decimals) for b in res.params],
        'Std.Err.': [fmt(s, decimals) for s in res.bse],
        't-value': [fmt(t, decimals) for t in res.tvalues],
        'p-value': [fmt_p(p, decimals) for p in res.pvalues],
    }, index=res.params.index)
    legend = ''
    if show_signif:
        table['Signif.'] = ['' if name in CONST_NAMES else signif_stars(p)
                            for name, p in res.pvalues.items()]
        legend = '\n\n' + SIGNIF_LEGEND
    align = ('left',) + ('right',) * 4 + (('left',) if show_signif else ())
    display(Markdown(f'\n\n{title}\n\n' + table.to_markdown(colalign=align, disable_numparse=True) + legend + '\n\n'))


def print_significant_coeffs(res, sign_level=0.05, theme='note', title=None):
    """Рамка со списком коэффициентов, значимых на уровне sign_level (без константы)."""
    names = [name for name, p in res.pvalues.items() if p < sign_level and name not in CONST_NAMES]
    if title is None:
        title = f'Коэффициенты, значимые на уровне {sign_level * 100:g}% ({errors_text(res, "dat")})'
    text = ', '.join(f'`{name}`' for name in names) if names else 'Значимых коэффициентов не выявлено.'
    display_answer(text, theme=theme, title=title)


def show_models(models, names=None, decimals=3, regressor_order=(), errors_note=None, info=None):
    """Сравнение регрессий (summary_col): строка Dep. Variable, коды значимости, пояснение про скобки.

    info — словарь {название строки: функция от подгонки} для дополнительных строк (N, F и т. п.).
    """
    table = summary_col(models, model_names=names, stars=True, float_format=f'%0.{decimals}f',
                        regressor_order=list(regressor_order), info_dict=info or {}).tables[0]
    top = pd.DataFrame({col: [m.model.endog_names] for col, m in zip(table.columns, models)},
                       index=['Dep. Variable'])
    table = pd.concat([top, table])
    if errors_note is None:
        errors_note = errors_text(models[0])
    note = f'{SIGNIF_LEGEND}. В скобках — {errors_note}.'
    display(Markdown('\n\n' + table.to_markdown(disable_numparse=True) + '\n\n' + note + '\n\n'))


def show_ci(res, level=0.95, decimals=3):
    """Доверительные интервалы: «Переменная | Нижняя граница | Верхняя граница», заголовок над таблицей."""
    ci = res.conf_int(alpha=1 - level)
    table = pd.DataFrame({'Переменная': ci.index,
                          'Нижняя граница': [fmt(x, decimals) for x in ci[0]],
                          'Верхняя граница': [fmt(x, decimals) for x in ci[1]]})
    title = f'**{level * 100:g}%-е доверительные интервалы ({errors_text(res)}):**'
    display(Markdown(f'\n\n{title}\n\n'
                     + table.to_markdown(index=False, colalign=('left', 'right', 'right'), disable_numparse=True) + '\n\n'))

# Загрузка данных
sleep75 = pd.read_csv('../datasets/sleep75.csv')
Labour = pd.read_csv('../datasets/Labour.csv')
Electricity = pd.read_csv('../datasets/Electricity.csv')


# Помощники листка (в общей части их нет)

def show_coefs_exact(res, decimals=2):
    """Как общий show_coefs, но tabulate не разбирает числа: конечные нули сохраняются (0.40)."""
    table = pd.DataFrame({'Переменная': res.params.index,
                          'Оценка': [fmt(b, decimals) for b in res.params]})
    text = table.to_markdown(index=False, colalign=('left', 'right'), disable_numparse=True)
    display(Markdown('\n\n' + text + '\n\n' + nobs_text(res) + '\n\n'))


def show_obs(res, ind, source=None, decimals=3):
    """Факт, предсказание и остаток для наблюдений с номерами ind (счёт с нуля, iloc).

    source — имя исходной таблицы: для подвыборок добавляется столбец с её индексом.
    """
    fact = res.model.endog[ind]
    fact_decimals = 0 if (fact % 1 == 0).all() else decimals  # целые — без дробной части
    table = pd.DataFrame({'№': [str(i) for i in ind]})
    if source:
        table[f'Индекс в {source}'] = [str(i) for i in res.model.data.row_labels[ind]]
    table['Факт'] = [fmt(x, fact_decimals) for x in fact]
    table['Предсказание'] = [fmt(x, decimals) for x in res.fittedvalues.iloc[ind]]
    table['Остаток'] = [fmt(x, decimals) for x in res.resid.iloc[ind]]
    text = table.to_markdown(index=False, colalign=('right',) * table.shape[1],
                             disable_numparse=True)
    display(Markdown('\n\n' + text + '\n\n'))


def show_sums(res, decimals=2):
    """TSS, ESS, RSS — одна рамка «Суммы квадратов»."""
    sums = {'TSS': res.centered_tss, 'ESS': res.ess, 'RSS': res.ssr}
    text = '\n\n'.join(f'${name} = {fmt(value, decimals)}$' for name, value in sums.items())
    display_answer(text, title='Суммы квадратов')


def show_r2(res, decimals=4):
    """R^2 и скорректированный R^2 — одна рамка «Коэффициент детерминации»."""
    text = (f'$R^2 = {fmt(res.rsquared, decimals)}$, '
            f'$R^2_{{adj}} = {fmt(res.rsquared_adj, decimals)}$')
    display_answer(text, title='Коэффициент детерминации')


def show_sigma(res, decimals=2):
    """Стандартная ошибка регрессии в рамке."""
    display_answer(fmt(np.sqrt(res.scale), decimals), title='Стандартная ошибка регрессии')

::: {.callout-important title="Важно"}
- Везде log — натуральный логарифм; константа включается в модель по умолчанию.
- Наборы данных `sleep75`, `Labour`, `Electricity` лежат в папке `datasets/`, их описание — в `datasets/data-description.md`.
- Наблюдения нумеруются с нуля (`iloc`); в задачах на подвыборках — порядковые номера внутри подвыборки.
- Имена коэффициентов — как в statsmodels: `Intercept` — константа, `np.log(x)` — $\log x$, `I(x ** 2)` — $x^2$.
- Округление указано в каждом вопросе; ответы считаются по неокруглённым оценкам. Оценки, которые при округлении обращаются в ноль, приведены с тремя значащими цифрами. Расхождение с ответом на единицу последнего знака — результат округления.
:::

# Суммы квадратов, остатки, предсказанные значения, коэффициент детерминации

## sleep equation #1

Для набора данных `sleep75` рассмотрим регрессию **sleep на totwrk, age, male, smsa**. Переменные `sleep` и `totwrk` измерены в минутах в неделю.

In [2]:
ind = [0, 3, 5, 9, 507, 699]
mod = fit_ols('sleep ~ totwrk + age + male + smsa', sleep75)

Найдите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

In [3]:
#| output: asis
show_coefs_exact(mod, decimals=2)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |  3494.22 |
| totwrk       |    -0.17 |
| age          |     2.81 |
| male         |    86.91 |
| smsa         |   -75.29 |

Модель была подогнана по 706 наблюдениям.



Для наблюдений с номерами 0, 3, 5, 9, 507, 699 (счёт с нуля, `iloc`) найдите фактическое значение зависимой переменной, предсказанное значение и остаток. **Ответ округлите до 3-х десятичных знаков.**

In [4]:
#| output: asis
show_obs(mod, ind)



|   № |   Факт |   Предсказание |   Остаток |
|----:|-------:|---------------:|----------:|
|   0 |   3113 |       3094.454 |    18.546 |
|   3 |   3083 |       2943.580 |   139.420 |
|   5 |   4063 |       3494.143 |   568.857 |
|   9 |   3018 |       3111.983 |   -93.983 |
| 507 |   3630 |       3476.914 |   153.086 |
| 699 |   2993 |       3295.398 |  -302.398 |



Вычислите TSS, ESS, RSS. **Ответ округлите до 2-х десятичных знаков.**

In [5]:
#| output: asis
show_sums(mod)



::: {.callout-note title="Суммы квадратов" icon="true"}
$TSS = 139239835.76$

$ESS = 17188382.23$

$RSS = 122051453.53$
:::



Вычислите $R^2$ и $R^2_{adj}$. **Ответ округлите до 4-х десятичных знаков.**

In [6]:
#| output: asis
show_r2(mod)



::: {.callout-note title="Коэффициент детерминации" icon="true"}
$R^2 = 0.1234$, $R^2_{adj} = 0.1184$
:::



Вычислите стандартную ошибку регрессии. **Ответ округлите до 2-х десятичных знаков.**

In [7]:
#| output: asis
show_sigma(mod)



::: {.callout-note title="Стандартная ошибка регрессии" icon="true"}
417.27
:::



## sleep equation #2

Для набора данных `sleep75` рассмотрим регрессию **sleep на totwrk, age, male, south** *только для жителей мегаполиса (smsa = 1)*.

In [8]:
ind = [0, 8, 55, 75, 99]
mod = fit_ols('sleep ~ totwrk + age + male + south', sleep75[sleep75.smsa == 1])

Найдите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

In [9]:
#| output: asis
show_coefs_exact(mod, decimals=2)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |  3460.43 |
| totwrk       |    -0.18 |
| age          |     2.28 |
| male         |    92.65 |
| south        |   168.85 |

Модель была подогнана по 282 наблюдениям.



Для наблюдений с порядковыми номерами внутри подвыборки 0, 8, 55, 75, 99 (счёт с нуля, `iloc`, после отбора строк с smsa = 1) найдите фактическое значение зависимой переменной, предсказанное значение и остаток. **Ответ округлите до 3-х десятичных знаков.**

In [10]:
#| output: asis
show_obs(mod, ind, source='sleep75')



|   № |   Индекс в sleep75 |   Факт |   Предсказание |   Остаток |
|----:|-------------------:|-------:|---------------:|----------:|
|   0 |                  2 |   2670 |       3132.491 |  -462.491 |
|   8 |                 19 |   3090 |       3370.083 |  -280.083 |
|  55 |                149 |   2985 |       2835.360 |   149.640 |
|  75 |                194 |   3660 |       3248.540 |   411.460 |
|  99 |                259 |   2618 |       2863.070 |  -245.070 |



Вычислите TSS, ESS, RSS. **Ответ округлите до 2-х десятичных знаков.**

In [11]:
#| output: asis
show_sums(mod)



::: {.callout-note title="Суммы квадратов" icon="true"}
$TSS = 50055043.40$

$ESS = 7396259.39$

$RSS = 42658784.01$
:::



Вычислите $R^2$ и $R^2_{adj}$. **Ответ округлите до 4-х десятичных знаков.**

In [12]:
#| output: asis
show_r2(mod)



::: {.callout-note title="Коэффициент детерминации" icon="true"}
$R^2 = 0.1478$, $R^2_{adj} = 0.1355$
:::



Вычислите стандартную ошибку регрессии. **Ответ округлите до 2-х десятичных знаков.**

In [13]:
#| output: asis
show_sigma(mod)



::: {.callout-note title="Стандартная ошибка регрессии" icon="true"}
392.43
:::



## sleep equation #3

Для набора данных `sleep75` рассмотрим регрессию **sleep на totwrk, age, smsa, south** *только для мужчин (male = 1)*.

In [14]:
ind = [0, 8, 44, 186, 279]
mod = fit_ols('sleep ~ totwrk + age + smsa + south', sleep75[sleep75.male == 1])

Найдите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

In [15]:
#| output: asis
show_coefs_exact(mod, decimals=2)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |  3600.96 |
| totwrk       |    -0.19 |
| age          |     3.30 |
| smsa         |   -77.69 |
| south        |    60.11 |

Модель была подогнана по 400 наблюдениям.



Для наблюдений с порядковыми номерами внутри подвыборки 0, 8, 44, 186, 279 (счёт с нуля, `iloc`, после отбора строк с male = 1) найдите фактическое значение зависимой переменной, предсказанное значение и остаток. **Ответ округлите до 3-х десятичных знаков.**

In [16]:
#| output: asis
show_obs(mod, ind, source='sleep75')



|   № |   Индекс в sleep75 |   Факт |   Предсказание |   Остаток |
|----:|-------------------:|-------:|---------------:|----------:|
|   0 |                  0 |   3113 |       3061.481 |    51.519 |
|   8 |                  9 |   3018 |       3087.341 |   -69.341 |
|  44 |                 48 |   3363 |       3268.521 |    94.479 |
| 186 |                262 |   3390 |       3228.976 |   161.024 |
| 279 |                440 |   2955 |       3399.060 |  -444.060 |



Вычислите TSS, ESS, RSS. **Ответ округлите до 2-х десятичных знаков.**

In [17]:
#| output: asis
show_sums(mod)



::: {.callout-note title="Суммы квадратов" icon="true"}
$TSS = 75570140.58$

$ESS = 12125135.60$

$RSS = 63445004.98$
:::



Вычислите $R^2$ и $R^2_{adj}$. **Ответ округлите до 4-х десятичных знаков.**

In [18]:
#| output: asis
show_r2(mod)



::: {.callout-note title="Коэффициент детерминации" icon="true"}
$R^2 = 0.1604$, $R^2_{adj} = 0.1519$
:::



Вычислите стандартную ошибку регрессии. **Ответ округлите до 2-х десятичных знаков.**

In [19]:
#| output: asis
show_sigma(mod)



::: {.callout-note title="Стандартная ошибка регрессии" icon="true"}
400.77
:::



## sleep equation #4

Для набора данных `sleep75` рассмотрим регрессию **sleep на totwrk, age, smsa, south** *только для женщин (male = 0)*.

In [20]:
ind = [0, 8, 44, 186, 279]
mod = fit_ols('sleep ~ totwrk + age + smsa + south', sleep75[sleep75.male == 0])

Найдите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

In [21]:
#| output: asis
show_coefs_exact(mod, decimals=2)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |  3468.20 |
| totwrk       |    -0.15 |
| age          |     1.65 |
| smsa         |   -37.69 |
| south        |   136.03 |

Модель была подогнана по 306 наблюдениям.



Для наблюдений с порядковыми номерами внутри подвыборки 0, 8, 44, 186, 279 (счёт с нуля, `iloc`, после отбора строк с male = 0) найдите фактическое значение зависимой переменной, предсказанное значение и остаток. **Ответ округлите до 3-х десятичных знаков.**

In [22]:
#| output: asis
show_obs(mod, ind, source='sleep75')



|   № |   Индекс в sleep75 |   Факт |   Предсказание |   Остаток |
|----:|-------------------:|-------:|---------------:|----------:|
|   0 |                  3 |   3083 |       2945.903 |   137.097 |
|   8 |                 67 |   2858 |       2920.662 |   -62.662 |
|  44 |                198 |   3730 |       3328.629 |   401.371 |
| 186 |                472 |   3376 |       3503.722 |  -127.722 |
| 279 |                628 |   3405 |       3215.073 |   189.927 |



Вычислите TSS, ESS, RSS. **Ответ округлите до 2-х десятичных знаков.**

In [23]:
#| output: asis
show_sums(mod)



::: {.callout-note title="Суммы квадратов" icon="true"}
$TSS = 63490152.12$

$ESS = 6300990.57$

$RSS = 57189161.55$
:::



Вычислите $R^2$ и $R^2_{adj}$. **Ответ округлите до 4-х десятичных знаков.**

In [24]:
#| output: asis
show_r2(mod)



::: {.callout-note title="Коэффициент детерминации" icon="true"}
$R^2 = 0.0992$, $R^2_{adj} = 0.0873$
:::



Вычислите стандартную ошибку регрессии. **Ответ округлите до 2-х десятичных знаков.**

In [25]:
#| output: asis
show_sigma(mod)



::: {.callout-note title="Стандартная ошибка регрессии" icon="true"}
435.89
:::



## output equation #1

Для набора данных `Labour` рассмотрим регрессию **output на capital, labour, wage**. Переменные `output` и `capital` измерены в млн евро, `wage` — в тыс. евро на работника.

In [26]:
ind = [1, 10, 23, 100, 348, 500]
mod = fit_ols('output ~ capital + labour + wage', Labour)

Найдите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

In [27]:
#| output: asis
show_coefs_exact(mod, decimals=2)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |   -16.75 |
| capital      |     0.40 |
| labour       |     0.05 |
| wage         |     0.44 |

Модель была подогнана по 569 наблюдениям.



Для наблюдений с номерами 1, 10, 23, 100, 348, 500 (счёт с нуля, `iloc`) найдите фактическое значение зависимой переменной, предсказанное значение и остаток. **Ответ округлите до 3-х десятичных знаков.**

In [28]:
#| output: asis
show_obs(mod, ind)



|   № |   Факт |   Предсказание |   Остаток |
|----:|-------:|---------------:|----------:|
|   1 |  3.664 |          0.457 |     3.207 |
|  10 |  3.246 |          5.666 |    -2.420 |
|  23 |  2.666 |         -0.625 |     3.292 |
| 100 |  2.181 |          6.952 |    -4.771 |
| 348 |  4.307 |          2.278 |     2.029 |
| 500 |  3.660 |         11.353 |    -7.694 |



Вычислите TSS, ESS, RSS. **Ответ округлите до 2-х десятичных знаков.**

In [29]:
#| output: asis
show_sums(mod)



::: {.callout-note title="Суммы квадратов" icon="true"}
$TSS = 2235073.29$

$ESS = 2190376.08$

$RSS = 44697.22$
:::



Вычислите $R^2$ и $R^2_{adj}$. **Ответ округлите до 4-х десятичных знаков.**

In [30]:
#| output: asis
show_r2(mod)



::: {.callout-note title="Коэффициент детерминации" icon="true"}
$R^2 = 0.9800$, $R^2_{adj} = 0.9799$
:::



Вычислите стандартную ошибку регрессии. **Ответ округлите до 2-х десятичных знаков.**

In [31]:
#| output: asis
show_sigma(mod)



::: {.callout-note title="Стандартная ошибка регрессии" icon="true"}
8.89
:::



## output equation #2

Для набора данных `Labour` рассмотрим регрессию **np.log(output) на np.log(capital), np.log(labour), np.log(wage)**.

In [32]:
ind = [1, 10, 23, 100, 348, 500]
mod = fit_ols('np.log(output) ~ np.log(capital) + np.log(labour) + np.log(wage)', Labour)

Найдите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

In [33]:
#| output: asis
show_coefs_exact(mod, decimals=2)



| Переменная      |   Оценка |
|:----------------|---------:|
| Intercept       |    -5.01 |
| np.log(capital) |     0.15 |
| np.log(labour)  |     0.72 |
| np.log(wage)    |     0.92 |

Модель была подогнана по 569 наблюдениям.



Для наблюдений с номерами 1, 10, 23, 100, 348, 500 (счёт с нуля, `iloc`) найдите фактическое значение зависимой переменной (здесь это `np.log(output)`), предсказанное значение и остаток. **Ответ округлите до 3-х десятичных знаков.**

In [34]:
#| output: asis
show_obs(mod, ind)



|   № |   Факт |   Предсказание |   Остаток |
|----:|-------:|---------------:|----------:|
|   1 |  1.299 |          1.347 |    -0.049 |
|  10 |  1.177 |          1.427 |    -0.250 |
|  23 |  0.981 |          0.344 |     0.637 |
| 100 |  0.780 |          1.006 |    -0.226 |
| 348 |  1.460 |          1.177 |     0.283 |
| 500 |  1.297 |          1.730 |    -0.433 |



Вычислите TSS, ESS, RSS. **Ответ округлите до 2-х десятичных знаков.**

In [35]:
#| output: asis
show_sums(mod)



::: {.callout-note title="Суммы квадратов" icon="true"}
$TSS = 797.67$

$ESS = 708.65$

$RSS = 89.02$
:::



Вычислите $R^2$ и $R^2_{adj}$. **Ответ округлите до 4-х десятичных знаков.**

In [36]:
#| output: asis
show_r2(mod)



::: {.callout-note title="Коэффициент детерминации" icon="true"}
$R^2 = 0.8884$, $R^2_{adj} = 0.8878$
:::



Вычислите стандартную ошибку регрессии. **Ответ округлите до 2-х десятичных знаков.**

In [37]:
#| output: asis
show_sigma(mod)



::: {.callout-note title="Стандартная ошибка регрессии" icon="true"}
0.40
:::



## cost equation #1

Для набора данных `Electricity` рассмотрим регрессию

`cost ~ q + I(q**2) + pk + pf + pl`

In [38]:
ind = [1, 16, 28, 47, 57, 93]
mod = fit_ols('cost ~ q + I(q**2) + pk + pf + pl', Electricity)

Найдите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

In [39]:
#| output: asis
show_coefs_exact(mod, decimals=2)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |   -81.36 |
| q            |  0.00432 |
| I(q ** 2)    | 1.57e-08 |
| pk           |     0.37 |
| pf           |     0.95 |
| pl           |  0.00361 |

Модель была подогнана по 158 наблюдениям.



Для наблюдений с номерами 1, 16, 28, 47, 57, 93 (счёт с нуля, `iloc`) найдите фактическое значение зависимой переменной, предсказанное значение и остаток. **Ответ округлите до 3-х десятичных знаков.**

In [40]:
#| output: asis
show_obs(mod, ind)



|   № |   Факт |   Предсказание |   Остаток |
|----:|-------:|---------------:|----------:|
|   1 |  3.043 |         -2.512 |     5.555 |
|  16 |  5.597 |          2.747 |     2.850 |
|  28 |  7.439 |         -2.269 |     9.709 |
|  47 | 28.786 |         34.228 |    -5.442 |
|  57 | 41.902 |         43.941 |    -2.040 |
|  93 |  9.467 |         -4.009 |    13.477 |



Вычислите TSS, ESS, RSS. **Ответ округлите до 2-х десятичных знаков.**

In [41]:
#| output: asis
show_sums(mod)



::: {.callout-note title="Суммы квадратов" icon="true"}
$TSS = 1189954.21$

$ESS = 1143993.96$

$RSS = 45960.25$
:::



Вычислите $R^2$ и $R^2_{adj}$. **Ответ округлите до 4-х десятичных знаков.**

In [42]:
#| output: asis
show_r2(mod)



::: {.callout-note title="Коэффициент детерминации" icon="true"}
$R^2 = 0.9614$, $R^2_{adj} = 0.9601$
:::



Вычислите стандартную ошибку регрессии. **Ответ округлите до 2-х десятичных знаков.**

In [43]:
#| output: asis
show_sigma(mod)



::: {.callout-note title="Стандартная ошибка регрессии" icon="true"}
17.39
:::



## cost equation #2

Для набора данных `Electricity` рассмотрим регрессию

```
np.log(cost) ~ np.log(q) + I(np.log(q)**2) +
    np.log(pk) + np.log(pf) + np.log(pl)
```

In [44]:
ind = [1, 16, 28, 47, 57, 93]
mod = fit_ols('np.log(cost) ~ np.log(q) + I(np.log(q)**2)'
              ' + np.log(pk) + np.log(pf) + np.log(pl)',
              Electricity)

Найдите коэффициенты подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

In [45]:
#| output: asis
show_coefs_exact(mod, decimals=2)



| Переменная        |   Оценка |
|:------------------|---------:|
| Intercept         |    -6.74 |
| np.log(q)         |     0.40 |
| I(np.log(q) ** 2) |     0.03 |
| np.log(pk)        |     0.16 |
| np.log(pf)        |     0.68 |
| np.log(pl)        |     0.15 |

Модель была подогнана по 158 наблюдениям.



Для наблюдений с номерами 1, 16, 28, 47, 57, 93 (счёт с нуля, `iloc`) найдите фактическое значение зависимой переменной (здесь это `np.log(cost)`), предсказанное значение и остаток. **Ответ округлите до 3-х десятичных знаков.**

In [46]:
#| output: asis
show_obs(mod, ind)



|   № |   Факт |   Предсказание |   Остаток |
|----:|-------:|---------------:|----------:|
|   1 |  1.113 |          1.452 |    -0.339 |
|  16 |  1.722 |          1.650 |     0.072 |
|  28 |  2.007 |          1.784 |     0.223 |
|  47 |  3.360 |          3.556 |    -0.196 |
|  57 |  3.735 |          3.846 |    -0.111 |
|  93 |  2.248 |          2.247 |     0.001 |



Вычислите TSS, ESS, RSS. **Ответ округлите до 2-х десятичных знаков.**

In [47]:
#| output: asis
show_sums(mod)



::: {.callout-note title="Суммы квадратов" icon="true"}
$TSS = 373.66$

$ESS = 370.76$

$RSS = 2.90$
:::



Вычислите $R^2$ и $R^2_{adj}$. **Ответ округлите до 4-х десятичных знаков.**

In [48]:
#| output: asis
show_r2(mod)



::: {.callout-note title="Коэффициент детерминации" icon="true"}
$R^2 = 0.9922$, $R^2_{adj} = 0.9920$
:::



Вычислите стандартную ошибку регрессии. **Ответ округлите до 2-х десятичных знаков.**

In [49]:
#| output: asis
show_sigma(mod)



::: {.callout-note title="Стандартная ошибка регрессии" icon="true"}
0.14
:::

